<p>
<font size='5' face='Georgia, Arial'>IIC-2233 Apunte Programación Avanzada</font><br>
<font size='1'>&copy; 2015 Karim Pichara - Christian Pieringer. Todos los derechos reservados. Editado por Equipo Docente IIC2233 2018-1 al 2026-2</font>
</p>

# Tabla de contenidos

1. [Ejecución de código de ejemplo del material que usa PyQt](#Ejecución-de-código-de-ejemplo-del-material-que-usa-PyQt)
2. [QTimer](#QTimer)
    1. [Reloj digital](#Reloj-digital)
    2. [Múltiples QTimer](#Múltiples-QTimer)
    3. [`singleShot`](#singleShot)
3. [Pausar y reanudar](#Pausar-y-reanudar)
4. [Loop de juego con un solo QTimer](#Loop-de-juego-con-un-solo-QTimer)

## Ejecución de código de ejemplo del material que usa PyQt

Jupyter no fue creado para ejecutar interfaces gráficas de escritorio, por lo que en este _notebook_ el código se muestra como texto y **no** se ejecuta. Cada ejemplo está en la carpeta `scripts/`, y te recomendamos **fuertemente** ejecutarlo desde tu computador e interactuar con él.

**Importante:** al ejecutar un _script_, la terminal debe estar posicionada en la misma carpeta del archivo `.py`. Por ejemplo:

```bash
cd scripts/2-qtimer
python3 1_reloj_digital.py
```

## QTimer

Un `QTimer` ejecuta un método **cada cierto intervalo de tiempo, dentro del _event loop_**. Es parecido al `threading.Timer` que vimos en la semana de *threading*, pero con dos diferencias clave:

- **No crea ningún _thread_.** Cuando se cumple el intervalo, el *event loop* ejecuta el método conectado a la señal `timeout` en el *thread* principal, igual que con un *click*. Por lo tanto, ese método **sí puede modificar la interfaz** y no hay problemas de concurrencia.
- **Se repite**: por defecto vuelve a ejecutarse cada vez que se cumple el intervalo, hasta que lo detengamos.

Como corre en el *thread* principal, el método conectado a `timeout` debe ser rápido, igual que cualquier *slot*. Si demora, congela la interfaz.

Los métodos más usados de `QTimer` (módulo `PyQt6.QtCore`) son:

| Método o señal | Descripción |
|---|---|
| `QTimer(parent)` | Crea el *timer*. Usar un *parent* (por ejemplo `self`) asegura que el objeto no sea eliminado mientras se usa. |
| `timeout` | Señal que se emite cada vez que se cumple el intervalo. |
| `setInterval(ms)` | Define el intervalo, en **milisegundos**. |
| `start()` / `start(ms)` | Inicia (o reinicia) el *timer*. La segunda forma también define el intervalo. |
| `stop()` | Detiene el *timer*. |
| `isActive()` | Retorna `True` si el *timer* está corriendo. |
| `setSingleShot(True)` | Hace que el *timer* se ejecute una sola vez. |
| `QTimer.singleShot(ms, metodo)` | Método estático: ejecuta `metodo` una sola vez después de `ms` milisegundos, sin crear un objeto. |

### Reloj digital

Partamos con un reloj que actualiza la hora cada 1 segundo. Este código se encuentra en el archivo `2-qtimer/1_reloj_digital.py`.

```python
from PyQt6.QtWidgets import QApplication, QWidget, QLabel, QVBoxLayout
from PyQt6.QtCore import QTimer
from PyQt6.QtGui import QFont
import sys
import datetime


class RelojDigital(QWidget):
    def __init__(self) -> None:
        super().__init__()

        # Crear label encargado de mostrar la hora
        self.label_timer = QLabel()
        self.label_timer.setFont(QFont("Times", 50))

        # Crear layout vertical para nuestro label
        layout = QVBoxLayout()
        layout.addWidget(self.label_timer)
        self.setLayout(layout)

        # Crear nuestro QTimer encargado de actualizar el tiempo cada 1 segundo
        timer = QTimer(self)
        timer.timeout.connect(self.mostrar_hora)
        timer.setInterval(1000)
        timer.start()

        # Definir título y tamaño ventana
        self.setWindowTitle("Reloj Digital con QTimer")
        self.setGeometry(100, 100, 250, 100)

        # Ejecutar el método para mostrar hora por primera vez
        self.mostrar_hora()

        # Mostrar ventana
        self.show()

    def mostrar_hora(self) -> None:
        # Obtener hora actual
        hora_actual = datetime.datetime.now().time()
        # Actualizar texto del label
        self.label_timer.setText(hora_actual.strftime("%H:%M:%S"))


if __name__ == "__main__":

    def hook(type_, value, traceback):
        # Imprimimos el error con el hook original de Python. Al instalar
        # nuestro propio sys.excepthook, PyQt no cierra el programa
        sys.__excepthook__(type_, value, traceback)

    sys.excepthook = hook
    app = QApplication([])
    reloj = RelojDigital()
    sys.exit(app.exec())
```

### Múltiples QTimer

Podemos tener varios `QTimer` funcionando al mismo tiempo, cada uno con su intervalo. En el siguiente ejemplo, cada `MiTimer` es un `QObject` que tiene su propio `QTimer` y avisa a la ventana mediante una señal. Al presionar el botón se crean cinco, con intervalos de 0,1 a 0,5 segundos. Cada uno cuenta hasta 9 y luego se detiene.

Este código se encuentra en el archivo `2-qtimer/2_multiples_qtimers.py`. En el *notebook* de *QThreads* veremos el mismo ejemplo hecho con *threads*: compáralos.

```python
import sys
from PyQt6.QtCore import pyqtSignal, QObject, QTimer
from PyQt6.QtWidgets import QApplication, QWidget, QLabel, QPushButton


class MiTimer(QObject):
    senal_actualizar = pyqtSignal(int, str)

    def __init__(self, indice: int, tiempo: float) -> None:
        super().__init__()
        self.indice = indice
        self.tiempo = tiempo
        self.indice_actual = 0
        self.timer = QTimer(self)

        # Acá se asigna el tiempo de duración del periodo entre ejecuciones
        self.timer.setInterval(int(tiempo * 1000))
        # Acá se conecta la subrutina que se ejecutará
        self.timer.timeout.connect(self.enviar_dato)

    def enviar_dato(self) -> None:
        if self.indice_actual <= 9:
            self.senal_actualizar.emit(self.indice, str(self.indice_actual))
            self.indice_actual += 1
        else:
            self.senal_actualizar.emit(self.indice, "Status: Qtimer terminado")
            self.timer.stop()

    def comenzar(self) -> None:
        self.timer.start()

    def sigue_andando(self) -> bool:
        return self.timer.isActive()


class MiVentana(QWidget):
    def __init__(self) -> None:
        super().__init__()
        self.timers = []
        self.init_gui()

    def init_gui(self) -> None:
        # Configuramos los widgets de la interfaz
        # Definimos un montón de labels que corresponderán a un Qtimer cada uno
        self.labels = {i: QLabel("Status: esperando Qtimer", self) for i in range(1, 6)}
        self.boton = QPushButton("Ejecutar Qtimers", self)
        self.boton.clicked.connect(self.ejecutar_timers)

        for i in range(1, 6):
            self.labels[i].setGeometry(10, (i - 1) * 30, 230, 30)

        self.boton.setGeometry(10, 150, 230, 30)
        # Configuramos las propiedades de la ventana.
        self.setWindowTitle("Ejemplo Qtimers")
        self.setGeometry(50, 50, 250, 200)
        self.show()

    def ejecutar_timers(self) -> None:
        """
        Este método crea cinco timers cada vez que se presiona el botón en la
        interfaz. Los timers recibirán como argumento el índice del label
        que les corresponde y el tiempo que toman entre cada iteración.
        """
        for timer in self.timers:
            if timer.sigue_andando():
                return

        self.timers = []
        for i in range(1, 6):
            timer = MiTimer(i, i / 10)
            # Se conecta la señal emitida por el timer a un método
            # de la ventana
            timer.senal_actualizar.connect(self.actualizar_labels)
            self.timers.append(timer)
            timer.comenzar()

    def actualizar_labels(self, indice: int, texto: str) -> None:
        """
        Este método actualiza el label correspondiente según los datos
        enviados desde un timer a través del índice y aplica el texto.
        """
        self.labels[indice].setText(texto)


if __name__ == "__main__":

    def hook(type_, value, traceback):
        # Imprimimos el error con el hook original de Python. Al instalar
        # nuestro propio sys.excepthook, PyQt no cierra el programa
        sys.__excepthook__(type_, value, traceback)

    sys.excepthook = hook
    app = QApplication([])
    ventana = MiVentana()
    sys.exit(app.exec())
```

### `singleShot`

A veces queremos ejecutar algo **una sola vez** después de cierto tiempo: mostrar un mensaje por 3 segundos, hacer que un enemigo aparezca más tarde, etc. Hay dos formas de hacerlo:

1. Crear un `QTimer` y llamar a `setSingleShot(True)` antes de iniciarlo. Útil si después lo queremos detener o reiniciar.
2. Usar el método estático `QTimer.singleShot(ms, metodo)`, sin crear un objeto. Es más corto, pero no se puede cancelar.

En el siguiente ejemplo, que extiende el reloj digital, la ventana se esconde a los 3 segundos (forma 1) y vuelve a aparecer a los 5 segundos (forma 2). Este código se encuentra en el archivo `2-qtimer/3_singleshot.py`.

```python
        # Forma 1: un QTimer configurado con setSingleShot(True).
        # Después de 3 segundos esconde la ventana, y no se vuelve a ejecutar
        self.timer_esconder = QTimer(self)
        self.timer_esconder.setSingleShot(True)
        self.timer_esconder.timeout.connect(self.hide)
        self.timer_esconder.setInterval(3000)
        self.timer_esconder.start()

        # Forma 2: el método estático QTimer.singleShot, sin crear un objeto.
        # Después de 5 segundos vuelve a mostrar la ventana
        QTimer.singleShot(5000, self.show)
```

Prueba comentar la línea `self.timer_esconder.setSingleShot(True)`: el *timer* volverá a esconder la ventana cada 3 segundos. Como `QTimer.singleShot` la muestra una sola vez (a los 5 segundos), desde los 6 segundos la ventana queda escondida para siempre, pero el programa **sigue corriendo**. Para terminarlo, cierra la terminal o presiona `Ctrl+\` (en Linux y macOS). `Ctrl+C` no sirve: la interrupción llega como una excepción dentro de un *slot*, y nuestro *hook* la imprime y deja el programa funcionando.

## Pausar y reanudar

Como un `QTimer` es el "motor" que hace avanzar algo en el tiempo, pausar es simplemente detenerlo con `stop()`, y reanudar es volver a iniciarlo con `start()`. El estado (en este caso, las décimas de segundo) vive en atributos del objeto, así que no se pierde.

Este código se encuentra en el archivo `2-qtimer/4_pausa.py`.

```python
class Cronometro(QWidget):
    def __init__(self) -> None:
        super().__init__()
        # Décimas de segundo transcurridas
        self.decimas = 0

        # Este QTimer avanza el cronómetro cada 100 ms. No lo iniciamos aún
        self.timer = QTimer(self)
        self.timer.setInterval(100)
        self.timer.timeout.connect(self.avanzar)

        self.init_gui()

    def init_gui(self) -> None:
        # Configuramos los widgets de la interfaz
        self.label = QLabel(self)
        self.label.setFont(QFont("Monospace", 30))
        self.label.setGeometry(10, 10, 230, 50)

        self.boton_pausa = QPushButton("Iniciar", self)
        self.boton_pausa.setGeometry(10, 70, 110, 30)
        self.boton_pausa.clicked.connect(self.pausar_o_reanudar)

        self.boton_reiniciar = QPushButton("Reiniciar", self)
        self.boton_reiniciar.setGeometry(130, 70, 110, 30)
        self.boton_reiniciar.clicked.connect(self.reiniciar)

        # Configuramos las propiedades de la ventana.
        self.setWindowTitle("Cronómetro con QTimer")
        self.setGeometry(50, 50, 250, 110)
        self.actualizar_label()
        self.show()

    def avanzar(self) -> None:
        self.decimas += 1
        self.actualizar_label()

    def pausar_o_reanudar(self) -> None:
        # isActive() nos dice si el QTimer está corriendo.
        # Pausar es simplemente detener el QTimer, y reanudar es
        # volver a iniciarlo. El estado (self.decimas) no se pierde
        if self.timer.isActive():
            self.timer.stop()
            self.boton_pausa.setText("Reanudar")
        else:
            self.timer.start()
            self.boton_pausa.setText("Pausar")

    def reiniciar(self) -> None:
        self.timer.stop()
        self.decimas = 0
        self.boton_pausa.setText("Iniciar")
        self.actualizar_label()

    def actualizar_label(self) -> None:
        minutos, decimas = divmod(self.decimas, 600)
        self.label.setText(f"{minutos:02d}:{decimas / 10:04.1f}")
```

En un juego, pausar significa detener **todos** los *timers* que hacen avanzar el juego, por lo que conviene tenerlos todos accesibles, por ejemplo en una lista.

Un detalle: al reanudar con `start()`, el intervalo parte de cero. Para intervalos cortos (como estos 100 ms) no se nota, pero si un *timer* de 10 segundos se pausa cuando le faltaba 1 segundo, al reanudar esperará 10 segundos completos. Si eso importa, `remainingTime()` te dice cuántos milisegundos faltaban antes de detenerlo.

## Loop de juego con un solo QTimer

En un juego o simulación con muchos elementos en movimiento, una opción muy cómoda es tener **un único `QTimer`** (el *tick* del juego) que en cada ejecución actualiza a **todos** los elementos. Comparado con tener un *timer* o un *thread* por elemento:

- Todos los elementos avanzan en el mismo instante, por lo que es fácil detectar colisiones o comparar posiciones.
- Pausar es detener un solo *timer*.
- Todo ocurre en el *thread* principal, así que no hay problemas de concurrencia.

En el siguiente ejemplo, 30 pelotas rebotan dentro de la ventana. `Pelota` es una clase común (no es un *thread* ni un `QObject`): solo guarda su posición y velocidad. El *back-end* (`Juego`) las mueve todas en cada *tick* y avisa al *front-end* (`Ventana`) con señales. La barra espaciadora pausa y reanuda el juego. En `keyPressEvent` usamos `isAutoRepeat()` para reaccionar solo cuando la tecla se acaba de presionar, y no muchas veces seguidas si se mantiene presionada (lo vemos con más detalle en el [*notebook* 5](5-ejemplo-aplicado-y-recetario.ipynb#isAutoRepeat-en-keyPressEvent)).

Este código se encuentra en el archivo `2-qtimer/5_loop_de_juego.py`.

```python
import sys
from random import randint
from PyQt6.QtCore import QObject, QTimer, Qt, pyqtSignal
from PyQt6.QtGui import QColor, QKeyEvent, QPixmap
from PyQt6.QtWidgets import QApplication, QWidget, QLabel

ANCHO, ALTO = 500, 400
TAMAÑO = 20
CANTIDAD_PELOTAS = 30
TICK_MS = 16  # ~60 actualizaciones por segundo


class Pelota:
    """
    Una pelota es solo datos: posición y velocidad. No es un thread ni un
    QObject, porque no hace nada por sí misma. Quien la mueve es el Juego.
    """

    def __init__(self, id_pelota: int) -> None:
        self.id = id_pelota
        self.x = randint(0, ANCHO - TAMAÑO)
        self.y = randint(0, ALTO - TAMAÑO)
        self.vx = randint(-4, 4) or 1
        self.vy = randint(-4, 4) or 1

    def mover(self) -> None:
        self.x += self.vx
        self.y += self.vy
        # Rebotamos en los bordes de la ventana
        if not 0 <= self.x <= ANCHO - TAMAÑO:
            self.vx *= -1
        if not 0 <= self.y <= ALTO - TAMAÑO:
            self.vy *= -1


class Juego(QObject):
    """
    Back-end. Un único QTimer hace avanzar a TODAS las pelotas en cada tick.
    """

    senal_mover = pyqtSignal(int, int, int)  # id, x, y
    senal_pausa = pyqtSignal(bool)  # True si quedó en pausa

    def __init__(self) -> None:
        super().__init__()
        self.pelotas = [Pelota(i) for i in range(CANTIDAD_PELOTAS)]
        self.timer = QTimer(self)
        self.timer.setInterval(TICK_MS)
        self.timer.timeout.connect(self.tick)

    def iniciar(self) -> None:
        self.timer.start()

    def tick(self) -> None:
        # En cada tick actualizamos el estado de todo el juego y avisamos
        # al front-end. Todo ocurre en el thread principal: no hay
        # problemas de concurrencia que cuidar
        for pelota in self.pelotas:
            pelota.mover()
            self.senal_mover.emit(pelota.id, pelota.x, pelota.y)

    def pausar_o_reanudar(self) -> None:
        if self.timer.isActive():
            self.timer.stop()
        else:
            self.timer.start()
        self.senal_pausa.emit(not self.timer.isActive())


class Ventana(QWidget):
    """
    Front-end. Solo dibuja lo que le dice el back-end y le avisa
    cuando el usuario presiona la barra espaciadora.
    """

    senal_tecla_pausa = pyqtSignal()

    def __init__(self) -> None:
        super().__init__()
        self.setWindowTitle("Loop de juego con un QTimer (Espacio: pausa)")
        self.setFixedSize(ANCHO, ALTO)
        self.setStyleSheet("background: #1e1e2e;")

        self.labels = {}
        for i in range(CANTIDAD_PELOTAS):
            pixmap = QPixmap(TAMAÑO, TAMAÑO)
            pixmap.fill(QColor(randint(80, 255), randint(80, 255), randint(80, 255)))
            label = QLabel(self)
            label.setPixmap(pixmap)
            label.setGeometry(-TAMAÑO, -TAMAÑO, TAMAÑO, TAMAÑO)
            self.labels[i] = label

        self.label_pausa = QLabel("PAUSA", self)
        self.label_pausa.setStyleSheet("color: white; font-size: 40px;")
        self.label_pausa.setGeometry(ANCHO // 2 - 70, ALTO // 2 - 30, 140, 60)
        self.label_pausa.hide()

    def mover(self, id_pelota: int, x: int, y: int) -> None:
        self.labels[id_pelota].move(x, y)

    def mostrar_pausa(self, en_pausa: bool) -> None:
        self.label_pausa.setVisible(en_pausa)

    def keyPressEvent(self, evento: QKeyEvent) -> None:
        if evento.key() == Qt.Key.Key_Space and not evento.isAutoRepeat():
            self.senal_tecla_pausa.emit()


if __name__ == "__main__":

    def hook(type_, value, traceback):
        # Imprimimos el error con el hook original de Python. Al instalar
        # nuestro propio sys.excepthook, PyQt no cierra el programa
        sys.__excepthook__(type_, value, traceback)

    sys.excepthook = hook
    app = QApplication([])
    juego = Juego()
    ventana = Ventana()

    # Conectamos back-end y front-end
    juego.senal_mover.connect(ventana.mover)
    juego.senal_pausa.connect(ventana.mostrar_pausa)
    ventana.senal_tecla_pausa.connect(juego.pausar_o_reanudar)

    ventana.show()
    juego.iniciar()
    sys.exit(app.exec())
```

Fíjate en que `Ventana` no sabe nada del `QTimer` ni de cómo se mueven las pelotas: solo dibuja lo que le dice el *back-end* y le avisa cuando se presiona la barra espaciadora. Es la misma separación *front-end*/*back-end* de la semana pasada.